# 01｜GTEx数据审计、标准化与逐染色体选样

本Notebook从旧的16 kb tissue-aligned GTEx+control元数据和已经完成的AlphaGenome 11模态分数开始，生成`GTEx_self/data/`中的冻结输入。

样本单位是 **variant–target gene–target tissue/context**。正样本为GTEx `PIP>0.9`；负样本同时保留GTEx `PIP<0.01`和matched control。代码会逐项检查旧坐标、Tissue、标签和染色体split，不修改任何上游文件。

In [ ]:
# ======================== 用户统一配置（优先只修改本单元） ========================
from pathlib import Path

PROJECT_ROOT = Path('/vepfs-mlp2/xts001/400107/code/AG_classification/GTEx_self')

# 旧GTEx元数据和旧AG 11模态评分结果：只读，不会覆盖。
SOURCE_METADATA_FILE = Path('/vepfs-mlp2/xts001/400107/data/AG_classification/tissue_aligned_16kb/tissue_aligned_gtex_control.parquet')
SOURCE_AG_SCORE_FILE = Path('/vepfs-mlp2/xts001/400107/results/AG_classification/scores_tissue_aligned_16kb/tissue_aligned_scores_11scorer.parquet')
GENCODE_REFERENCE_FILE = Path('/vepfs-mlp2/xts001/400107/code/alphagen/model/reference/hg38/gencode.v46.annotation.gtf.gz.feather')

# 新项目中的正样本、负样本、统一主表和实际选样文件。
POSITIVE_FILE = PROJECT_ROOT / 'data/positive_gtex_pip_gt_0p9_scored_11modal.parquet'
NEGATIVE_FILE = PROJECT_ROOT / 'data/negative_gtex_pip_lt_0p01_plus_control_scored_11modal.parquet'
UNIFIED_FILE = PROJECT_ROOT / 'data/unified_binary_dataset.parquet'
SELECTED_SAMPLES_FILE = PROJECT_ROOT / 'data/selected_samples.parquet'
CHROMOSOME_BALANCE_FILE = PROJECT_ROOT / 'data/chromosome_balance.csv'

# Train设为None表示按冻结染色体保留原始全部样本，不做类别下采样。
# Valid逐染色体执行严格负/正比例；Test保留全部样本，再构造不丢样本的1:1评估轮次。
TRAIN_NEG_PER_POS = None
VALID_NEG_PER_POS = 1
TEST_NEG_PER_POS = None
INSUFFICIENT_NEGATIVE_POLICY = 'downsample_positive'  # 也可设为'error'

# 下列配置与训练Notebook保持同名，便于复制。01中只记录，不启动训练。
FEATURE_RECIPE = 'score33_plus_tissue'
MODELS_TO_RUN = ['LogisticRegression', 'RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM', 'DeepMLP']
GPU_DEVICE = 'cuda:0'
RANDOM_SEED = 20260817
RUN_TAG = 'smoke_rf_20260817'
FAST_MODE = True

RUN_DIR = PROJECT_ROOT / 'results' / RUN_TAG
print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUN_DIR      =', RUN_DIR)

## 固定科学定义

- AG输入长度固定为16,384 bp。
- Train/Valid/Test只能按下面的冻结染色体划分，禁止随机行划分。
- Train和Test按冻结染色体保留原始全部正负样本，不做类别下采样。
- Valid的1:1平衡在每条染色体内部独立完成；负样本不足时按明示规则下采样正样本。
- Test预测覆盖全部样本；额外构造穷尽式1:1评估轮次，多数类跨轮次不重复。

In [ ]:
import hashlib
import json
import os
import re
from collections import OrderedDict

import numpy as np
import pandas as pd
from IPython.display import display

MODALITIES = [
    'ATAC', 'DNASE', 'CHIP_TF', 'CHIP_HISTONE', 'CAGE', 'PROCAP',
    'RNA_SEQ', 'CONTACT_MAPS', 'SPLICE_SITES', 'SPLICE_SITE_USAGE',
    'SPLICE_JUNCTIONS',
]

SPLIT_CHROMOSOMES = OrderedDict([
    ('train', ['chr1', 'chr4', 'chr7', 'chr8', 'chr10', 'chr13', 'chr15']),
    ('valid', ['chr2', 'chr5', 'chr11', 'chr14', 'chr17', 'chr20', 'chr22', 'chrX']),
    ('test',  ['chr3', 'chr6', 'chr9', 'chr12', 'chr16', 'chr18', 'chr19', 'chr21']),
])
CHROMOSOME_TO_SPLIT = {
    chromosome: split
    for split, chromosomes in SPLIT_CHROMOSOMES.items()
    for chromosome in chromosomes
}

FORBIDDEN_CLASSIFIER_FEATURES = [
    'label', 'label_reason', 'sample_source', 'sample_group', 'negative_source',
    'PIP', 'Beta', 'SE', 'CS size', 'CS Unique ID',
    'variant_key', 'chromosome', 'position', 'reference', 'alternate',
    'rf_split', 'split', 'sample_id', 'source_excel_row',
    'target_gene_id', 'gene_match_mode', 'tissue_match_mode',
    'Variant Selection Type', 'selection_types_all',
    'control_context_group', 'tissue_assignment',
    'match_rule', 'winning_track', 'winning_gene', 'n_candidates',
    'distance', 'pair', 'loc_offset_bp',
    'gene_tss', 'tss_distance', 'tss_mapping_method',
]

def normalize_chromosome(value):
    text = str(value).strip()
    if not text.startswith('chr'):
        text = 'chr' + text
    return text

def normalize_tissue(value):
    # 统一空格、连字符和标点；保留可读的GTEx大小写。
    text = '' if pd.isna(value) else str(value).strip()
    text = re.sub(r'[^A-Za-z0-9]+', '_', text)
    return re.sub(r'_+', '_', text).strip('_')

def atomic_parquet(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name('.' + path.name + '.tmp')
    frame.to_parquet(temporary, index=False, compression='zstd')
    os.replace(temporary, path)

def stable_sha256(values):
    digest = hashlib.sha256()
    for value in sorted(map(str, values)):
        digest.update(value.encode('utf-8'))
        digest.update(b'\n')
    return digest.hexdigest()

# 路径审计：缺少任何输入时立即停止。
path_audit = []
for name, path in {
    'SOURCE_METADATA_FILE': SOURCE_METADATA_FILE,
    'SOURCE_AG_SCORE_FILE': SOURCE_AG_SCORE_FILE,
    'GENCODE_REFERENCE_FILE': GENCODE_REFERENCE_FILE,
}.items():
    path_audit.append({
        'name': name,
        'path': str(path),
        'exists': path.exists(),
        'size_gib': path.stat().st_size / 1024**3 if path.exists() else np.nan,
    })
path_audit = pd.DataFrame(path_audit)
display(path_audit)
if not path_audit['exists'].all():
    raise FileNotFoundError(path_audit.loc[~path_audit.exists, 'path'].tolist())

In [ ]:
# ======================== 读取旧数据并核对一对一身份字段 ========================
metadata = pd.read_parquet(SOURCE_METADATA_FILE)
scores = pd.read_parquet(SOURCE_AG_SCORE_FILE)

required_meta = {
    'sample_id', 'variant_key', 'chromosome', 'position', 'reference', 'alternate',
    'target_gene_id', 'target_tissue', 'label', 'sample_source', 'PIP',
    'rf_split', 'context_length',
}
required_score = {
    'sample_id', 'variant_key', 'label', 'sample_source', 'target_tissue',
    'target_gene_id', 'rf_split', 'context_length', *MODALITIES,
}
missing_meta = sorted(required_meta - set(metadata.columns))
missing_score = sorted(required_score - set(scores.columns))
if missing_meta or missing_score:
    raise ValueError({'missing_metadata': missing_meta, 'missing_scores': missing_score})
if metadata['sample_id'].duplicated().any() or scores['sample_id'].duplicated().any():
    raise ValueError('旧metadata或score存在重复sample_id')
if set(metadata.sample_id) != set(scores.sample_id):
    raise ValueError('旧metadata与score的sample_id集合不一致')

# 对两个上游文件共享的关键字段逐项比对，防止错误join。
identity_columns = [
    'variant_key', 'label', 'sample_source', 'target_tissue',
    'target_gene_id', 'rf_split', 'context_length',
]
identity = metadata[['sample_id'] + identity_columns].merge(
    scores[['sample_id'] + identity_columns], on='sample_id', how='inner',
    suffixes=('__meta', '__score'), validate='one_to_one',
)
identity_mismatch = {}
for column in identity_columns:
    left = identity[f'{column}__meta'].astype('string').fillna('<NA>')
    right = identity[f'{column}__score'].astype('string').fillna('<NA>')
    identity_mismatch[column] = int(left.ne(right).sum())
if any(identity_mismatch.values()):
    raise ValueError({'upstream_identity_mismatch': identity_mismatch})

audit_score_columns = [column for column in scores.columns if '__' in column]
score_payload = scores[['sample_id'] + MODALITIES + audit_score_columns].copy()
unified = metadata.merge(score_payload, on='sample_id', how='inner', validate='one_to_one')
print('metadata rows:', len(metadata), 'score rows:', len(scores), 'unified rows:', len(unified))
print('identity mismatch:', identity_mismatch)

In [ ]:
# ======================== 坐标、Tissue、标签和旧split标准化 ========================
unified['chromosome'] = unified['chromosome'].map(normalize_chromosome)
unified['position'] = pd.to_numeric(unified['position'], errors='raise').astype('int64')
unified['reference'] = unified['reference'].astype(str).str.upper()
unified['alternate'] = unified['alternate'].astype(str).str.upper()
unified['target_tissue'] = unified['target_tissue'].astype('string')
unified['target_tissue_standardized'] = unified['target_tissue'].map(normalize_tissue).astype('string')
unified['label'] = pd.to_numeric(unified['label'], errors='raise').astype('int8')

if unified.target_tissue_standardized.eq('').any():
    raise ValueError('存在空target_tissue')
if not unified.context_length.eq(16_384).all():
    raise ValueError('存在非16,384 bp的旧记录')
if not unified.chromosome.isin(CHROMOSOME_TO_SPLIT).all():
    bad = sorted(unified.loc[~unified.chromosome.isin(CHROMOSOME_TO_SPLIT), 'chromosome'].unique())
    raise ValueError({'noncanonical_chromosomes': bad})

reconstructed_key = (
    unified.chromosome.astype(str) + ':' + unified.position.astype(str) + ':'
    + unified.reference + ':' + unified.alternate
)
coordinate_key_mismatch = int(reconstructed_key.ne(unified.variant_key.astype(str)).sum())
if coordinate_key_mismatch:
    raise ValueError({'coordinate_variant_key_mismatch': coordinate_key_mismatch})

expected_split = unified.chromosome.map(CHROMOSOME_TO_SPLIT)
old_split_mismatch = int(expected_split.ne(unified.rf_split.astype(str)).sum())
if old_split_mismatch:
    raise ValueError({'old_rf_split_mismatch': old_split_mismatch})
unified['rf_split'] = expected_split.astype('string')

# 严格复核三类旧样本的标签定义。
is_positive = unified.sample_source.eq('GTEx_positive')
is_gtex_negative = unified.sample_source.eq('GTEx_negative')
is_control = unified.sample_source.eq('control_negative')
if not (is_positive | is_gtex_negative | is_control).all():
    raise ValueError(unified.loc[~(is_positive | is_gtex_negative | is_control), 'sample_source'].value_counts().to_dict())
if not (unified.loc[is_positive, 'label'].eq(1) & unified.loc[is_positive, 'PIP'].gt(0.9)).all():
    raise ValueError('GTEx positive不完全满足label=1且PIP>0.9')
if not (unified.loc[is_gtex_negative, 'label'].eq(0) & unified.loc[is_gtex_negative, 'PIP'].lt(0.01)).all():
    raise ValueError('GTEx negative不完全满足label=0且PIP<0.01')
if not unified.loc[is_control, 'label'].eq(0).all():
    raise ValueError('control不完全满足label=0')

unified['sample_group'] = np.where(unified.label.eq(1), 'positive', 'negative')
unified['negative_source'] = np.select(
    [is_gtex_negative, is_control], ['gtex_pip_lt_0p01', 'matched_control'], default=pd.NA,
)

# 正负sample_id必须互斥；物理variant允许在不同gene/tissue context中标签不同，但必须显式报告。
positive_ids = set(unified.loc[unified.label.eq(1), 'sample_id'])
negative_ids = set(unified.loc[unified.label.eq(0), 'sample_id'])
duplicated_sample_ids_between_labels = len(positive_ids & negative_ids)
conflicting_variant_keys = (
    unified.groupby('variant_key', sort=False)['label'].nunique().loc[lambda s: s.gt(1)].index.astype(str)
)
variant_cross_split = int(unified.groupby('variant_key').rf_split.nunique().gt(1).sum())
if duplicated_sample_ids_between_labels:
    raise ValueError('正负样本之间存在重复sample_id')
if variant_cross_split:
    raise ValueError('同一物理variant跨越不同染色体split')

print('coordinate_variant_key_mismatch =', coordinate_key_mismatch)
print('old_rf_split_mismatch            =', old_split_mismatch)
print('conflicting_variant_keys         =', len(conflicting_variant_keys))
print('variant_cross_split              =', variant_cross_split)

In [ ]:
# ======================== 用目标Ensembl gene精确回填hg38 TSS ========================
# TSS只用于Test分区评估，明确禁止作为分类特征。
gencode_columns = [
    'Chromosome', 'Feature', 'Start', 'End', 'Strand',
    'gene_id', 'gene_id_nopatch',
]
genes = pd.read_feather(GENCODE_REFERENCE_FILE, columns=gencode_columns)
genes = genes.loc[genes.Feature.eq('gene')].copy()
genes['chromosome'] = genes.Chromosome.astype('string')
genes['target_gene_id_normalized'] = genes.gene_id_nopatch.fillna(
    genes.gene_id.astype('string').str.split('.').str[0]
).astype('string')

# 该Feather的Start为0-based、End为1-based闭坐标：
# 正链TSS使用Start+1，负链TSS使用End，最终均转成hg38 1-based。
genes['gene_tss'] = np.where(
    genes.Strand.eq('-'), genes.End, genes.Start + 1
).astype('int64')
tss_reference = genes[[
    'target_gene_id_normalized', 'chromosome', 'gene_tss',
]].drop_duplicates()
tss_multiplicity = (
    tss_reference.groupby(['target_gene_id_normalized', 'chromosome'])
    .gene_tss.nunique()
)
unique_tss_keys = tss_multiplicity.loc[tss_multiplicity.eq(1)].index
tss_reference = (
    tss_reference.set_index(['target_gene_id_normalized', 'chromosome'])
    .loc[unique_tss_keys].reset_index()
    .drop_duplicates(['target_gene_id_normalized', 'chromosome'])
)
tss_lookup = {
    (str(gene_id), str(chromosome)): int(gene_tss)
    for gene_id, chromosome, gene_tss in tss_reference.itertuples(index=False, name=None)
}

normalized_target_gene = unified.target_gene_id.astype('string').str.split('.').str[0]
tss_keys = zip(normalized_target_gene, unified.chromosome.astype('string'))
mapped_tss = [
    tss_lookup.get((str(gene_id), str(chromosome)))
    if pd.notna(gene_id) else None
    for gene_id, chromosome in tss_keys
]
unified['gene_tss'] = pd.Series(mapped_tss, index=unified.index, dtype='Int64')
unified['tss_distance'] = (
    unified.position.astype('Int64') - unified.gene_tss
).abs().astype('Float64')
unified['tss_mapping_method'] = np.where(
    unified.gene_tss.notna(),
    'gencode_v46_exact_ensembl_id_same_chromosome',
    'unresolved_target_gene',
)

tss_mapping_audit = (
    unified.groupby(['rf_split', 'label', 'sample_source', 'tss_mapping_method'], dropna=False)
    .size().rename('contexts').reset_index()
)
tss_coverage = (
    unified.assign(tss_mapped=unified.gene_tss.notna())
    .groupby(['rf_split', 'label', 'sample_source'], observed=True)
    .agg(contexts=('sample_id', 'size'), tss_mapped=('tss_mapped', 'sum'))
    .reset_index()
)
tss_coverage['coverage_pct'] = 100.0 * tss_coverage.tss_mapped / tss_coverage.contexts
tss_mapping_audit.to_csv(PROJECT_ROOT / 'data/tss_mapping_audit.csv', index=False)
tss_coverage.to_csv(PROJECT_ROOT / 'data/tss_coverage.csv', index=False)
print('目标gene TSS覆盖率')
display(tss_coverage)

In [ ]:
# ======================== 保存标准化正表、负表和统一主表 ========================
positive = unified.loc[unified.label.eq(1)].copy()
negative = unified.loc[unified.label.eq(0)].copy()
if len(positive) + len(negative) != len(unified):
    raise AssertionError('正负表未覆盖全部统一主表')

# 新文件按稳定键排序，便于哈希审计和版本比较。
sort_columns = ['rf_split', 'chromosome', 'position', 'sample_id']
positive = positive.sort_values(sort_columns, kind='stable').reset_index(drop=True)
negative = negative.sort_values(sort_columns, kind='stable').reset_index(drop=True)
unified = unified.sort_values(sort_columns, kind='stable').reset_index(drop=True)

atomic_parquet(positive, POSITIVE_FILE)
atomic_parquet(negative, NEGATIVE_FILE)
atomic_parquet(unified, UNIFIED_FILE)

conflict_table = unified.loc[unified.variant_key.astype(str).isin(set(conflicting_variant_keys))].copy()
conflict_table.to_csv(PROJECT_ROOT / 'data/conflicting_variant_contexts.csv', index=False)

standardized_counts = pd.DataFrame([
    {'table': 'positive', 'rows': len(positive), 'unique_variants': positive.variant_key.nunique()},
    {'table': 'negative', 'rows': len(negative), 'unique_variants': negative.variant_key.nunique()},
    {'table': 'unified', 'rows': len(unified), 'unique_variants': unified.variant_key.nunique()},
])
display(standardized_counts)

## 逐染色体平衡规则

Train和Test中的每条染色体保留全部原始样本，不强制目标比例。Valid中的每条染色体分别执行：

1. 目标比例定义为`negative / positive`，默认严格1:1；
2. 若负样本足够，保留该染色体全部正样本并确定性下采样负样本；
3. 若负样本不足且策略为`downsample_positive`，保留可用负样本并下采样正样本；
4. 采样顺序由`RANDOM_SEED + sample_id`的SHA-256确定，不依赖原文件行顺序；
5. Train/Test分别硬检查其`sample_id`集合与冻结主表完全一致；
6. Test的1:1仅作为模型锁定后的评估轮次：多数类不放回切块，少数类按需复用，所有Test样本至少覆盖一次。

In [ ]:
def deterministic_take(frame, n, salt):
    if n > len(frame):
        raise ValueError({'requested': n, 'available': len(frame), 'salt': salt})
    ranked = frame.assign(
        _selection_hash=frame.sample_id.astype(str).map(
            lambda value: hashlib.sha256(
                f'{RANDOM_SEED}|{salt}|{value}'.encode('utf-8')
            ).hexdigest()
        )
    ).sort_values(['_selection_hash', 'sample_id'], kind='stable')
    return ranked.head(n).drop(columns='_selection_hash')

def select_one_chromosome(frame, split, chromosome, neg_per_pos):
    pos = frame.loc[frame.label.eq(1)].copy()
    neg = frame.loc[frame.label.eq(0)].copy()
    n_pos, n_neg = len(pos), len(neg)
    if n_pos == 0 or n_neg == 0:
        raise ValueError(f'{split}/{chromosome}缺少一个类别: pos={n_pos}, neg={n_neg}')

    if neg_per_pos is None:
        if split not in {'train', 'test'}:
            raise ValueError('只有Train/Test允许NEG_PER_POS=None')
        selected = frame.copy()
        selected['selection_target_neg_per_pos'] = np.nan
        selected['selection_policy'] = f'keep_original_{split}'
        return selected

    if not isinstance(neg_per_pos, int) or neg_per_pos < 1:
        raise ValueError('Valid/Test的NEG_PER_POS必须为正整数')

    required_neg_for_all_positive = n_pos * neg_per_pos
    if n_neg >= required_neg_for_all_positive:
        keep_pos = n_pos
    elif INSUFFICIENT_NEGATIVE_POLICY == 'downsample_positive':
        keep_pos = n_neg // neg_per_pos
    elif INSUFFICIENT_NEGATIVE_POLICY == 'error':
        raise ValueError(
            f'{split}/{chromosome}负样本不足: pos={n_pos}, neg={n_neg}, '
            f'目标neg/pos={neg_per_pos}'
        )
    else:
        raise ValueError(f'未知INSUFFICIENT_NEGATIVE_POLICY={INSUFFICIENT_NEGATIVE_POLICY}')

    if keep_pos < 1:
        raise ValueError(f'{split}/{chromosome}平衡后没有可用正样本')
    keep_neg = keep_pos * neg_per_pos
    selected_pos = deterministic_take(pos, keep_pos, f'{split}|{chromosome}|positive')
    selected_neg = deterministic_take(neg, keep_neg, f'{split}|{chromosome}|negative')
    selected = pd.concat([selected_pos, selected_neg], ignore_index=True, sort=False)
    selected['selection_target_neg_per_pos'] = neg_per_pos
    selected['selection_policy'] = INSUFFICIENT_NEGATIVE_POLICY
    return selected

ratio_by_split = {
    'train': TRAIN_NEG_PER_POS,
    'valid': VALID_NEG_PER_POS,
    'test': TEST_NEG_PER_POS,
}
if TEST_NEG_PER_POS is not None:
    raise ValueError('Test主表必须保留全部样本，TEST_NEG_PER_POS只能为None')

selected_parts = []
for split, chromosomes in SPLIT_CHROMOSOMES.items():
    for chromosome in chromosomes:
        block = unified.loc[
            unified.rf_split.eq(split) & unified.chromosome.eq(chromosome)
        ].copy()
        selected_parts.append(
            select_one_chromosome(block, split, chromosome, ratio_by_split[split])
        )
selected = pd.concat(selected_parts, ignore_index=True, sort=False)
selected = selected.sort_values(sort_columns, kind='stable').reset_index(drop=True)
if selected.sample_id.duplicated().any():
    raise AssertionError('selected_samples出现重复sample_id')
for preserved_split in ['train', 'test']:
    source_ids = set(unified.loc[unified.rf_split.eq(preserved_split), 'sample_id'])
    selected_ids = set(selected.loc[selected.rf_split.eq(preserved_split), 'sample_id'])
    if source_ids != selected_ids:
        raise AssertionError(
            f'{preserved_split}未完整保留冻结染色体中的原始sample_id'
        )

def chromosome_balance(frame):
    counts = (
        frame.groupby(['rf_split', 'chromosome', 'label'], observed=True)
        .size().unstack(fill_value=0).rename(columns={0: 'negative', 1: 'positive'})
        .reset_index()
    )
    for column in ['positive', 'negative']:
        if column not in counts:
            counts[column] = 0
    counts['actual_neg_per_pos'] = counts.negative / counts.positive
    counts['target_neg_per_pos'] = counts.rf_split.map(ratio_by_split)
    counts['balance_mode'] = np.where(
        counts.rf_split.isin(['train', 'test']), 'keep_original', 'fixed_ratio'
    )
    counts['ratio_exact'] = np.where(
        counts.rf_split.isin(['train', 'test']),
        True,
        np.isclose(counts.actual_neg_per_pos, counts.target_neg_per_pos),
    )
    counts['total'] = counts.positive + counts.negative
    split_rank = {'train': 0, 'valid': 1, 'test': 2}
    counts['_split_rank'] = counts.rf_split.map(split_rank)
    counts['_chrom_rank'] = counts.chromosome.str.replace('chr', '', regex=False).replace('X', '23').astype(int)
    return counts.sort_values(['_split_rank', '_chrom_rank']).drop(columns=['_split_rank', '_chrom_rank'])

balance = chromosome_balance(selected)
if not balance.ratio_exact.all():
    raise AssertionError(balance.loc[~balance.ratio_exact].to_dict('records'))
if not balance.loc[balance.rf_split.eq('valid'), 'actual_neg_per_pos'].eq(1).all():
    raise AssertionError('Valid并非每条染色体1:1')

RUN_DIR.mkdir(parents=True, exist_ok=True)
atomic_parquet(selected, SELECTED_SAMPLES_FILE)
atomic_parquet(selected, RUN_DIR / 'selected_samples.parquet')
balance.to_csv(CHROMOSOME_BALANCE_FILE, index=False)
balance.to_csv(RUN_DIR / 'chromosome_balance.csv', index=False)
display(balance)

In [ ]:
# ======================== 保存并展示最终审计表 ========================
split_counts = (
    selected.groupby(['rf_split', 'label'], observed=True).size()
    .unstack(fill_value=0).rename(columns={0: 'negative', 1: 'positive'}).reset_index()
)
split_counts['actual_neg_per_pos'] = split_counts.negative / split_counts.positive

tissue_counts = (
    selected.groupby(['target_tissue_standardized', 'label'], observed=True).size()
    .unstack(fill_value=0).rename(columns={0: 'negative', 1: 'positive'}).reset_index()
)
for column in ['positive', 'negative']:
    if column not in tissue_counts:
        tissue_counts[column] = 0
tissue_counts['total'] = tissue_counts.positive + tissue_counts.negative

coverage_rows = []
for split, split_frame in selected.groupby('rf_split', observed=True):
    for modality in MODALITIES:
        non_missing = int(pd.to_numeric(split_frame[modality], errors='coerce').notna().sum())
        coverage_rows.append({
            'rf_split': split, 'modality': modality, 'rows': len(split_frame),
            'non_missing': non_missing, 'coverage_pct': 100.0 * non_missing / len(split_frame),
        })
modality_coverage = pd.DataFrame(coverage_rows)

source_counts = (
    selected.groupby(['rf_split', 'sample_source', 'label'], observed=True)
    .size().rename('rows').reset_index()
)
unique_variant_counts = (
    selected.groupby(['rf_split', 'label'], observed=True).variant_key.nunique()
    .rename('unique_variants').reset_index()
)

tissue_counts.to_csv(PROJECT_ROOT / 'data/tissue_counts_selected.csv', index=False)
modality_coverage.to_csv(PROJECT_ROOT / 'data/modality_coverage_selected.csv', index=False)
source_counts.to_csv(PROJECT_ROOT / 'data/source_counts_selected.csv', index=False)
unique_variant_counts.to_csv(PROJECT_ROOT / 'data/unique_variant_counts_selected.csv', index=False)

print('每个split的实际数量和比例')
display(split_counts)
print('每个Tissue的正负样本数量')
display(tissue_counts)
print('11种模态覆盖率')
display(modality_coverage)
print('唯一物理variant数量')
display(unique_variant_counts)
print('负样本来源仅用于审计，禁止作为模型输入')
display(source_counts)

In [ ]:
# ======================== 冻结数据摘要和哈希 ========================
summary = {
    'status': 'complete_validated',
    'classification_unit': 'variant-target gene-target tissue/context',
    'source_metadata_file': str(SOURCE_METADATA_FILE),
    'source_ag_score_file': str(SOURCE_AG_SCORE_FILE),
    'gencode_reference_file': str(GENCODE_REFERENCE_FILE),
    'context_length': 16_384,
    'label_definition': {
        'positive': 'GTEx PIP > 0.9',
        'negative_gtex': 'GTEx PIP < 0.01',
        'negative_control': 'matched control from the frozen upstream dataset',
    },
    'standardized_files': {
        'positive': str(POSITIVE_FILE),
        'negative': str(NEGATIVE_FILE),
        'unified': str(UNIFIED_FILE),
        'selected': str(SELECTED_SAMPLES_FILE),
        'chromosome_balance': str(CHROMOSOME_BALANCE_FILE),
    },
    'counts': {
        'unified_rows': int(len(unified)),
        'positive_rows': int(len(positive)),
        'negative_rows': int(len(negative)),
        'selected_rows': int(len(selected)),
        'unified_unique_variants': int(unified.variant_key.nunique()),
        'selected_unique_variants': int(selected.variant_key.nunique()),
        'target_tissues': int(unified.target_tissue_standardized.nunique()),
        'tss_mapped_rows': int(unified.gene_tss.notna().sum()),
    },
    'conflicts': {
        'duplicate_sample_id_between_labels': int(duplicated_sample_ids_between_labels),
        'variant_keys_with_multiple_context_labels': int(len(conflicting_variant_keys)),
        'variant_keys_crossing_splits': int(variant_cross_split),
    },
    'coordinate_and_split_audit': {
        'coordinate_variant_key_mismatch': int(coordinate_key_mismatch),
        'old_rf_split_mismatch': int(old_split_mismatch),
    },
    'balance_rule': {
        'train_neg_per_pos': TRAIN_NEG_PER_POS,
        'train_balance_mode': 'keep_original_by_frozen_chromosome',
        'valid_neg_per_pos': VALID_NEG_PER_POS,
        'test_neg_per_pos': TEST_NEG_PER_POS,
        'test_balance_mode': 'keep_all_then_exhaustive_balanced_evaluation_rounds',
        'insufficient_negative_policy': INSUFFICIENT_NEGATIVE_POLICY,
        'sampling': 'deterministic SHA256 order within each chromosome and label',
    },
    'modalities': MODALITIES,
    'forbidden_classifier_features': FORBIDDEN_CLASSIFIER_FEATURES,
    'sample_id_sha256': {
        'positive': stable_sha256(positive.sample_id),
        'negative': stable_sha256(negative.sample_id),
        'unified': stable_sha256(unified.sample_id),
        'selected': stable_sha256(selected.sample_id),
    },
}
summary_path = PROJECT_ROOT / 'data/dataset_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n01 Notebook已完成。正式六模型训练尚未启动。')